<a href="https://colab.research.google.com/github/kuds/reinforce-tactics/blob/main/notebooks/ppo_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

# Force headless pygame BEFORE anything imports pygame (directly or
# transitively). SDL picks its video driver at import time, so
# setting this later is a no-op for an already-loaded SDL backend.
os.environ.setdefault("SDL_VIDEODRIVER", "dummy")

# Reinforce Tactics — PPO Training

This notebook trains a **MaskablePPO** agent against the **`RandomBot`**
opponent on the 6×6 beginner map. It uses an SB3 `EvalCallback`-style
periodic-eval pattern: a single `model.learn()` call drives `TOTAL_TIMESTEPS`
of training, and the eval callback runs `N_EVAL_EPISODES` evaluation
episodes every `EVAL_FREQ` env steps. Defaults: 5,000,000 total steps,
30 episodes every 100,000 steps (= 50 evaluations across the run).

We use the random opponent as the **starting curriculum**: it is weak enough
that the agent can experience early wins (and the +win terminal reward)
within only a few thousand steps. Once the agent reliably beats `RandomBot`,
graduate to the harder `SimpleBot` opponent by setting `OPPONENT = "simple"`
in the config cell.

At each evaluation the agent is scored on:
- **Win rate** (% of games won against the configured opponent)
- **Average episode reward**
- **Average episode length** (steps)

The best-so-far model (by eval win rate) is saved to
`BENCHMARK_DIR / "best_model.zip"` automatically, and the final model is
saved to `final_model.zip`.

**Runtime:** GPU recommended (L4 or better). CPU is possible but slower.

---

### Why MaskablePPO?

The game has a `MultiDiscrete` action space where many action combinations
are invalid at any given time (e.g. you can't attack a tile with no enemy).
**Action masking** prevents the agent from sampling these invalid actions,
which typically yields 2–3× faster convergence compared to plain PPO.


## 1. Setup

In [ ]:
# Install dependencies
!pip install -q gymnasium stable-baselines3 sb3-contrib tensorboard pandas numpy torch matplotlib opencv-python-headless

import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Clone repo and install as a package
import os
import sys
from pathlib import Path

REPO_DIR = Path("reinforce-tactics")
if REPO_DIR.exists():
    os.chdir(REPO_DIR)
elif Path("notebooks").exists():
    # Already inside the repo
    os.chdir("..")
else:
    print("Cloning repository...")
    !git clone https://github.com/kuds/reinforce-tactics.git
    os.chdir(REPO_DIR)

# Install the package so all imports resolve
!pip install -q -e .

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f"Working directory: {os.getcwd()}")

## 2. Imports

In [ ]:
import json
import time
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sb3_contrib import MaskablePPO

from reinforcetactics.rl.masking import make_maskable_env, make_maskable_vec_env

print("All imports successful.")

## 2b. Storage configuration

Choose where to save benchmark outputs. Set `USE_GOOGLE_DRIVE = True` to
persist results to Google Drive (recommended for Colab -- files survive
runtime disconnects). Set `False` to use the default local/Colab storage.

Each execution saves outputs under a **per-opponent, datetime-stamped subfolder**
(e.g. `benchmarks/ppo_vs_random/20250615_143022/`), so results are organized
by opponent and previous runs are preserved automatically.

In [ ]:
# --- Storage configuration ---
# Set to True to save all outputs (checkpoints, results, plots) to Google Drive.
# This is recommended when running on Google Colab, since files on the local
# Colab filesystem are lost when the runtime disconnects.
# Set to False to use the default local/Colab storage.
USE_GOOGLE_DRIVE = True

# Google Drive base directory (only used when USE_GOOGLE_DRIVE is True).
# This path is relative to your Google Drive root (MyDrive/).
# The final output path is: {DRIVE_BASE_DIR}/ppo_vs_{OPPONENT}/{datetime}/
DRIVE_BASE_DIR = "reinforce-tactics/benchmarks"

# --- Mount Google Drive if enabled ---
if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive

        drive.mount("/content/drive")
        SAVE_DIR = Path("/content/drive/MyDrive") / DRIVE_BASE_DIR
        SAVE_DIR.mkdir(parents=True, exist_ok=True)
        print("Google Drive mounted successfully.")
        print(f"Save directory: {SAVE_DIR}")
    except ImportError:
        print("WARNING: google.colab not available (not running in Colab).")
        print("  Falling back to local storage.")
        USE_GOOGLE_DRIVE = False
        SAVE_DIR = None
    except Exception as e:
        print(f"WARNING: Failed to mount Google Drive: {e}")
        print("  Falling back to local storage.")
        USE_GOOGLE_DRIVE = False
        SAVE_DIR = None
else:
    SAVE_DIR = None
    print("Using local storage (default).")
    print("  Tip: Set USE_GOOGLE_DRIVE = True to persist results to Google Drive.")

## 3. Configuration

In [ ]:
# --- Benchmark settings ---
MAP_FILE = "maps/1v1/beginner.csv"  # 6x6 beginner map
OPPONENT = "random"  # Start with random for easier wins
MAX_STEPS = 400  # max env steps per episode (raised so MAX_TURNS triggers first)
MAX_TURNS = 20  # max game turns before auto-draw (terminated, not truncated)
N_ENVS = 4  # parallel training envs
SEED = 42

# Action space mode:
#   'flat_discrete'  — exact per-action masks (recommended, eliminates invalid actions)
#   'multi_discrete' — per-dimension masks (over-approximation, original behaviour)
ACTION_SPACE = "flat_discrete"

# Training & evaluation schedule (SB3 EvalCallback-style: total timesteps
# + a fixed eval frequency, replacing the older hand-picked checkpoint list).
TOTAL_TIMESTEPS = 5_000_000
EVAL_FREQ = 100_000  # run an evaluation every N env steps (across all envs)
N_EVAL_EPISODES = 30  # episodes per evaluation

# --- Reward configuration ---
# Tuned so capturing the enemy HQ dominates the alternative of farming
# kills against a respawning opponent. Old weights made `kill` (+10-15)
# worth more than a turn of `seize_progress` (+1), pushing the policy
# into a kill-farm local optimum that hit the step cap every episode.
REWARD_CONFIG = {
    # Terminal rewards
    "win": 1000.0,
    "loss": -1000.0,
    "draw": -200.0,
    # Potential-based shaping (small; only telescoped delta enters reward)
    "income_diff": 0.05,
    "unit_diff": 0.3,
    "structure_control": 1.0,
    # Per-action rewards — capture/seize now dominate combat farming
    "create_unit": 0.5,
    "move": 0.0,
    "damage_scale": 0.05,
    "kill": 5.0,
    "seize_progress": 5.0,
    "capture": 200.0,
    "cure": 5.0,
    "heal_scale": 0.5,
    "paralyze": 8.0,
    "haste": 6.0,
    "defence_buff": 5.0,
    "attack_buff": 5.0,
    # Penalties
    "invalid_action": -10.0,
    "turn_penalty": -2.0,
}

# PPO hyperparameters
PPO_CONFIG = dict(
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.05,  # Higher entropy for exploration
    vf_coef=0.5,
    max_grad_norm=0.5,
)

# --- Per-run output directory (opponent + datetime-stamped) ---
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_SUBDIR = f"ppo_vs_{OPPONENT}"

# Output paths — uses Google Drive if enabled, otherwise local storage.
# Each execution gets its own subfolder: ppo_vs_{OPPONENT}/{datetime}/
if SAVE_DIR is not None:
    BENCHMARK_DIR = SAVE_DIR / RUN_SUBDIR / RUN_ID
else:
    BENCHMARK_DIR = Path("benchmarks") / RUN_SUBDIR / RUN_ID
BENCHMARK_DIR.mkdir(parents=True, exist_ok=True)
CHARTS_DIR = BENCHMARK_DIR / "charts"
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Map:          {MAP_FILE}")
print(f"Opponent:     {OPPONENT}")
print(f"Action space: {ACTION_SPACE}")
print(f"Max steps:    {MAX_STEPS}")
print(f"Max turns:    {MAX_TURNS}")
print(f"Total steps:  {TOTAL_TIMESTEPS:,}")
print(f"Eval freq:    {EVAL_FREQ:,} env steps  ({TOTAL_TIMESTEPS // EVAL_FREQ} evaluations)")
print(f"Eval eps:     {N_EVAL_EPISODES}")
print(f"ent_coef:     {PPO_CONFIG['ent_coef']}")
print(f"turn_penalty: {REWARD_CONFIG['turn_penalty']}")
print(f"draw:         {REWARD_CONFIG['draw']}")
print(f"Storage:      {'Google Drive' if USE_GOOGLE_DRIVE else 'Local'}")
print(f"Run ID:       {RUN_ID}")
print(f"Output dir:   {BENCHMARK_DIR}")

## 4. Create environments

In [ ]:
# Training envs (vectorized, headless)
vec_env = make_maskable_vec_env(
    n_envs=N_ENVS,
    map_file=MAP_FILE,
    opponent=OPPONENT,
    max_steps=MAX_STEPS,
    max_turns=MAX_TURNS,
    reward_config=REWARD_CONFIG,
    seed=SEED,
    use_subprocess=False,  # DummyVecEnv (safer in notebooks)
    action_space_type=ACTION_SPACE,
)

# Single eval env, deterministic given SEED + episode index. The eval callback
# below feeds it through evaluate_model() at fixed timestep intervals.
eval_env = make_maskable_env(
    map_file=MAP_FILE,
    opponent=OPPONENT,
    max_steps=MAX_STEPS,
    max_turns=MAX_TURNS,
    reward_config=REWARD_CONFIG,
    action_space_type=ACTION_SPACE,
    seed=SEED,
)

print(f"Observation space: {vec_env.observation_space}")
print(f"Action space:      {vec_env.action_space}")


## 5. Create MaskablePPO model

In [ ]:
model = MaskablePPO(
    "MultiInputPolicy",
    vec_env,
    verbose=0,
    tensorboard_log=str(BENCHMARK_DIR / "tensorboard"),
    device=DEVICE,
    seed=SEED,
    **PPO_CONFIG,
)

print("MaskablePPO model created.")
print(f"Policy:  {model.policy.__class__.__name__}")
print(f"Device:  {model.device}")

## 6. Import evaluation helper

Use the shared evaluation function from the library.

In [ ]:
from reinforcetactics.rl.evaluation import evaluate_model

print("Using evaluate_model from reinforcetactics.rl.evaluation")

In [ ]:
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.utils import safe_mean


class TrainingMetricsCallback(BaseCallback):
    """Capture per-rollout PPO training metrics during ``model.learn()``.

    SB3's ``Logger.dump()`` clears ``name_to_value`` between rollouts, so
    rollout/* values are gone by the time the next callback hook fires. To
    work around this we capture rollout/* directly from
    ``self.model.ep_info_buffer`` at ``_on_rollout_end`` (the value source
    the logger itself uses) and read train/* from the logger at the *next*
    ``_on_rollout_start`` — by which point ``train()`` has run for the
    previous iteration and populated those keys. ``_on_training_end``
    picks up the final iteration that has no follow-up rollout.

    The accumulated records persist across multiple ``model.learn()``
    invocations, so the per-checkpoint training loop produces one
    continuous timeline.
    """

    TRAIN_KEYS = (
        "train/approx_kl",
        "train/clip_fraction",
        "train/entropy_loss",
        "train/explained_variance",
        "train/learning_rate",
        "train/loss",
        "train/policy_gradient_loss",
        "train/value_loss",
    )

    def __init__(self):
        super().__init__()
        self.records = []  # list of dicts: {timesteps, ...metrics}
        self._pending = None  # snapshot in progress

    def _on_step(self) -> bool:
        return True

    def _on_rollout_end(self) -> None:
        # Capture rollout stats directly from ep_info_buffer; the logger
        # clears them immediately after _dump_logs.
        snapshot = {"timesteps": self.num_timesteps}
        ep_buffer = getattr(self.model, "ep_info_buffer", None)
        if ep_buffer:
            rewards = [ep["r"] for ep in ep_buffer if "r" in ep]
            lengths = [ep["l"] for ep in ep_buffer if "l" in ep]
            if rewards:
                snapshot["rollout/ep_rew_mean"] = float(safe_mean(rewards))
            if lengths:
                snapshot["rollout/ep_len_mean"] = float(safe_mean(lengths))
        self._pending = snapshot

    def _commit_pending(self) -> None:
        """Merge train/* from the logger into the pending record and commit."""
        if self._pending is None:
            return
        for key in self.TRAIN_KEYS:
            val = self.model.logger.name_to_value.get(key)
            if val is not None:
                self._pending[key] = float(val)
        # Only emit records that contain at least one metric beyond timesteps.
        if len(self._pending) > 1:
            self.records.append(self._pending)
        self._pending = None

    def _on_rollout_start(self) -> None:
        # train() of the previous iteration has run by this hook, so
        # train/* values are now populated in the logger.
        self._commit_pending()

    def _on_training_end(self) -> None:
        self._commit_pending()



class PeriodicEvalCallback(BaseCallback):
    """Run ``evaluate_model`` every ``eval_freq`` env steps.

    Mirrors SB3's ``EvalCallback`` contract — a single ``model.learn()`` call
    drives evaluation at a fixed cadence — but captures full win / loss / draw
    breakdown via the project's ``evaluate_model`` helper (SB3's built-in
    callback only logs mean reward and episode length).

    The callback gates on ``num_timesteps`` (total env steps across all
    sub-envs) so the cadence is independent of ``n_envs``: ``eval_freq=25000``
    fires every 25,000 env steps regardless of how many parallel envs are
    rolling. Best model (by win rate) is saved to ``save_dir/best_model.zip``
    when ``save_dir`` is provided.
    """

    def __init__(
        self,
        eval_env,
        eval_freq: int,
        n_eval_episodes: int = 30,
        eval_seed_base: int = 0,
        save_dir=None,
        verbose: int = 1,
    ):
        super().__init__(verbose=verbose)
        self.eval_env = eval_env
        self.eval_freq = int(eval_freq)
        self.n_eval_episodes = int(n_eval_episodes)
        self.eval_seed_base = int(eval_seed_base)
        self.save_dir = Path(save_dir) if save_dir is not None else None
        self.results = []  # one dict per eval; same shape as evaluate_model()
        self.best_win_rate = -1.0
        self._last_eval_block = -1

    def _on_step(self) -> bool:
        # Trigger when num_timesteps crosses an eval_freq boundary. Using
        # block index (not modulo) avoids missing/double-firing when
        # num_timesteps jumps by n_envs > 1 each step.
        block = self.num_timesteps // self.eval_freq
        if block > self._last_eval_block:
            self._last_eval_block = block
            self._do_eval()
        return True

    def _do_eval(self) -> None:
        eval_seed = self.eval_seed_base + 1000 * self._last_eval_block
        m = evaluate_model(
            self.model,
            self.eval_env,
            n_episodes=self.n_eval_episodes,
            seed=eval_seed,
        )
        m["timesteps"] = int(self.num_timesteps)
        m["eval_seed"] = eval_seed
        self.results.append(m)

        # Tensorboard
        self.logger.record("eval/win_rate", m["win_rate"])
        self.logger.record("eval/mean_reward", m["avg_reward"])
        self.logger.record("eval/mean_ep_length", m["avg_length"])

        if self.verbose:
            print(
                f"  [eval @ {m['timesteps']:>9,}]  "
                f"WR={m['win_rate'] * 100:5.1f}%  "
                f"reward={m['avg_reward']:+8.1f} (+/-{m['std_reward']:5.1f})  "
                f"len={m['avg_length']:5.1f}  "
                f"W/L/D={m['wins']}/{m['losses']}/{m['draws']}"
            )

        # Save best by win rate (with mean reward as a tiebreaker so we don't
        # latch onto the first 0% checkpoint).
        if self.save_dir is not None:
            score = (m["win_rate"], m["avg_reward"])
            best = (self.best_win_rate, getattr(self, "_best_reward", float("-inf")))
            if score > best:
                self.best_win_rate = m["win_rate"]
                self._best_reward = m["avg_reward"]
                self.model.save(str(self.save_dir / "best_model.zip"))


metrics_callback = TrainingMetricsCallback()
eval_callback = PeriodicEvalCallback(
    eval_env,
    eval_freq=EVAL_FREQ,
    n_eval_episodes=N_EVAL_EPISODES,
    eval_seed_base=SEED,
    save_dir=BENCHMARK_DIR,
)
print("TrainingMetricsCallback + PeriodicEvalCallback ready.")


## 7. Train and evaluate at each checkpoint

We train incrementally: 0 → 10K → 50K → 200K → 1M → 2M timesteps,
evaluating at each checkpoint.

In [ ]:
# Single model.learn() call. The eval callback fires every EVAL_FREQ env
# steps and appends an entry to ``eval_callback.results`` — that list is what
# the downstream plots, the diagnose_training cell, and the saved JSON all
# read from (results = eval_callback.results below).
print(f"Training for {TOTAL_TIMESTEPS:,} timesteps "
      f"({TOTAL_TIMESTEPS // EVAL_FREQ} evaluations every {EVAL_FREQ:,} steps)...")

start_time = time.time()
model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    reset_num_timesteps=False,
    progress_bar=True,
    callback=[metrics_callback, eval_callback],
)
total_time = time.time() - start_time

results = eval_callback.results
model.save(str(BENCHMARK_DIR / "final_model.zip"))

print(f"\nTotal wall time: {total_time / 60:.1f} minutes")
print(f"Captured {len(results)} eval snapshots and "
      f"{len(metrics_callback.records)} training-metric snapshots.")
print(f"Best win rate during training: {eval_callback.best_win_rate * 100:.1f}%  "
      f"(saved to {BENCHMARK_DIR / 'best_model.zip'})")


## 8. Results table

In [ ]:
df = pd.DataFrame(results)
df["win_rate_pct"] = (df["win_rate"] * 100).round(1)
df["avg_reward"] = df["avg_reward"].round(1)
df["avg_length"] = df["avg_length"].round(1)

display_df = df[["timesteps", "win_rate_pct", "avg_reward", "avg_length", "wins", "losses", "draws"]].copy()
display_df.columns = ["Timesteps", "Win Rate (%)", "Avg Reward", "Avg Length", "Wins", "Losses", "Draws"]
display_df = display_df.set_index("Timesteps")

# With many evals (default schedule = 50) the table is short enough to show
# in full; if EVAL_FREQ is reduced, fall back to head + tail so the table
# stays readable. The full table is always in the saved CSV.
MAX_TABLE_ROWS = 30
if len(display_df) > MAX_TABLE_ROWS:
    half = MAX_TABLE_ROWS // 2
    print(f"Showing first {half} and last {half} of {len(display_df)} eval rows. Full table is in the saved CSV.")
    display_df = pd.concat([display_df.head(half), display_df.tail(half)])
display_df


## 9. Training curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ts = [r["timesteps"] for r in results]

# Win rate
ax = axes[0]
wr = [r["win_rate"] * 100 for r in results]
ax.plot(ts, wr, "o-", color="#2196F3", linewidth=2, markersize=8)
ax.set_xlabel("Timesteps")
ax.set_ylabel("Win Rate (%)")
ax.set_title(f"Win Rate vs {OPPONENT}")
ax.set_xscale("log")
ax.set_ylim(-5, 105)
ax.axhline(y=70, color="green", linestyle="--", alpha=0.5, label="70% target")
ax.legend()
ax.grid(True, alpha=0.3)

# Average reward
ax = axes[1]
avg_r = [r["avg_reward"] for r in results]
std_r = [r["std_reward"] for r in results]
ax.plot(ts, avg_r, "o-", color="#FF9800", linewidth=2, markersize=8)
ax.fill_between(ts, [a - s for a, s in zip(avg_r, std_r)], [a + s for a, s in zip(avg_r, std_r)], alpha=0.2, color="#FF9800")
ax.set_xlabel("Timesteps")
ax.set_ylabel("Average Reward")
ax.set_title("Average Episode Reward")
ax.set_xscale("log")
ax.grid(True, alpha=0.3)

# Episode length
ax = axes[2]
avg_l = [r["avg_length"] for r in results]
std_l = [r["std_length"] for r in results]
ax.plot(ts, avg_l, "o-", color="#4CAF50", linewidth=2, markersize=8)
ax.fill_between(ts, [a - s for a, s in zip(avg_l, std_l)], [a + s for a, s in zip(avg_l, std_l)], alpha=0.2, color="#4CAF50")
ax.set_xlabel("Timesteps")
ax.set_ylabel("Average Length (steps)")
ax.set_title("Average Episode Length")
ax.set_xscale("log")
ax.grid(True, alpha=0.3)

fig.suptitle(f"PPO Baseline Benchmarks  |  6x6 beginner map  |  vs {OPPONENT}", fontsize=13, fontweight="bold", y=1.02)
fig.tight_layout()

fig.savefig(str(CHARTS_DIR / "training_curves.png"), dpi=150, bbox_inches="tight")
print(f"Saved plot: {CHARTS_DIR / 'training_curves.png'}")
plt.show()

## 9a. Watch the Agent Play (Video Replay)

Record the trained agent playing a full game and view it as an inline
MP4 video. This uses **headless rendering** via `pygame-ce` — no display
server required (works on Colab).

The video shows the actual game map with terrain, structures, units,
health bars, and player colours exactly as they appear in the game UI.

In [ ]:
# pygame-ce is required for headless rendering. SDL_VIDEODRIVER=dummy was
# already set in the very first cell of the notebook; setting it again here
# would be a no-op once SDL has picked a backend.
try:
    import pygame

    print(f"pygame version: {pygame.ver}")
except ImportError:
    print("Installing pygame-ce...")
    !pip install -q pygame-ce
    import pygame

    print(f"pygame version: {pygame.ver}")

from reinforcetactics.utils.video import display_video_in_notebook, record_evaluation_to_video

print("Video recording utilities loaded.")


In [ ]:
# Sanity check before recording. If sprites still load with
# USE_PIXEL_ART=False below, the kernel is running stale code —
# Runtime > Disconnect and delete runtime, then re-run from the top.
import os

import pygame

from reinforcetactics.ui.renderer import Renderer

print("SDL_VIDEODRIVER:", os.environ.get("SDL_VIDEODRIVER"))
print("pygame display init:", pygame.display.get_init())

_probe_env = make_maskable_env(map_file=MAP_FILE, opponent=OPPONENT)
_probe_env.reset()
_r = Renderer(_probe_env.unwrapped.game_state, replay_mode=True, headless=True, pixel_art=False)
print("screen:", type(_r.screen).__name__, _r.screen.get_size())
print("tile sprites loaded:", sum(v is not None for v in _r.tile_images.values()))
print("unit sprites loaded:", len(_r.unit_images))
print("animator:", _r.animator)
_probe_env.close()

In [ ]:
# Record one full game of the trained agent
# This creates a fresh environment (not the eval_env used above) to avoid
# state leakage, then runs the model through a complete episode while
# capturing every frame with the headless renderer.

video_env = make_maskable_env(
    map_file=MAP_FILE,
    opponent=OPPONENT,
    max_steps=MAX_STEPS,
    max_turns=MAX_TURNS,
    reward_config=REWARD_CONFIG,
    action_space_type=ACTION_SPACE,
)

video_path = str(BENCHMARK_DIR / "agent_replay.mp4")

# Set to True to render with the bundled pixel-art sprites
# (assets/sprites/) instead of the default coloured rects + unit letters.
USE_PIXEL_ART = False

print("Recording agent gameplay...")
result = record_evaluation_to_video(
    env=video_env,
    model=model,
    output_path=video_path,
    fps=4,  # 4 fps — one action per 0.25s
    max_steps=MAX_STEPS,
    deterministic=True,
    use_pixel_art=USE_PIXEL_ART,
)

video_env.close()

winner_str = {1: "Agent wins", 2: "Opponent wins", None: "Draw"}
print(f"\nGame result:   {winner_str.get(result['winner'], 'Unknown')}")
print(f"Total reward:  {result['total_reward']:.1f}")
print(f"Steps taken:   {result['steps']}")
print(f"Video saved:   {result['video_path']}")

In [ ]:
# Display the video inline in the notebook
# On Colab this embeds the MP4 directly; locally it opens in the browser.
display_video_in_notebook(result["video_path"])

## 10. Save results

In [ ]:
# Save benchmark results as JSON
import subprocess


def _git_commit_id() -> str:
    """Return the current HEAD commit (short hash + dirty flag), or 'unknown'."""
    try:
        sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], stderr=subprocess.DEVNULL, text=True).strip()
        dirty = subprocess.call(["git", "diff", "--quiet", "HEAD"], stderr=subprocess.DEVNULL)
        return f"{sha}-dirty" if dirty else sha
    except (subprocess.CalledProcessError, FileNotFoundError):
        return "unknown"


benchmark_data = {
    "metadata": {
        "date": datetime.now().isoformat(),
        "git_commit": _git_commit_id(),
        "map": MAP_FILE,
        "opponent": OPPONENT,
        "max_steps": MAX_STEPS,
        "max_turns": MAX_TURNS,
        "n_envs": N_ENVS,
        "total_timesteps": TOTAL_TIMESTEPS,
        "eval_freq": EVAL_FREQ,
        "n_eval_episodes": N_EVAL_EPISODES,
        "seed": SEED,
        "device": DEVICE,
        "ppo_config": PPO_CONFIG,
        "reward_config": REWARD_CONFIG,
        "storage": "google_drive" if USE_GOOGLE_DRIVE else "local",
    },
    "results": results,
}

results_path = BENCHMARK_DIR / "benchmark_results.json"
with open(results_path, "w") as f:
    json.dump(benchmark_data, f, indent=2)

print(f"Saved results:  {results_path}")
print(f"Git commit:     {benchmark_data['metadata']['git_commit']}")

# Also save as CSV for easy viewing
csv_path = BENCHMARK_DIR / "benchmark_results.csv"
df.to_csv(csv_path, index=False)
print(f"Saved CSV:      {csv_path}")

# List all saved files
print("\nAll benchmark files:")
for p in sorted(BENCHMARK_DIR.rglob("*")):
    if p.is_file():
        size = p.stat().st_size
        if size > 1024 * 1024:
            size_str = f"{size / 1024 / 1024:.1f} MB"
        elif size > 1024:
            size_str = f"{size / 1024:.1f} KB"
        else:
            size_str = f"{size} B"
        rel = p.relative_to(BENCHMARK_DIR)
        print(f"  {str(rel):40s}  {size_str}")

if USE_GOOGLE_DRIVE:
    print("\nFiles are saved to Google Drive at:")
    print(f"  My Drive/{DRIVE_BASE_DIR}/{RUN_SUBDIR}/{RUN_ID}/")
    print("  These files will persist after the Colab runtime disconnects.")
else:
    print("\nFiles are saved to local storage at:")
    print(f"  {BENCHMARK_DIR.resolve()}")
    print("  WARNING: These files will be lost if the Colab runtime disconnects.")
    print("  Set USE_GOOGLE_DRIVE = True in the storage config cell to persist them.")

## 11. TensorBoard (optional)

Launch TensorBoard to inspect detailed training metrics (loss, entropy, etc.).

In [ ]:
# Uncomment to launch TensorBoard inline:
# %load_ext tensorboard
# %tensorboard --logdir benchmarks/ppo_vs_simplebot/tensorboard

print("To view TensorBoard locally, run:")
print(f"  tensorboard --logdir {BENCHMARK_DIR / 'tensorboard'}")

## 12. Interpreting the results

### What to expect (vs `RandomBot`)

`RandomBot` is intentionally weak (uniform sampling from legal actions),
so a well-trained agent should saturate quickly. With the default schedule
(5M timesteps / eval every 100K) you should see win rate climb from
~10–40% in the first few evals, cross 70% by ~200K-500K, and plateau
above ~90% well before the run completes. Once the curve plateaus, switch
`OPPONENT = "simple"` (`SimpleBot`) for a stiffer challenge — expect
lower starting win rates and a slower climb.

**Note:** Exact numbers depend on hardware and random seed. The important
thing is the curve has a similar *shape* — monotonically increasing win
rate with diminishing returns once the agent dominates.

### If your results differ significantly

- **Win rate stuck at 0%:** The agent has never experienced a +win
  reward. Check that the eval episode lengths aren't all hitting
  `MAX_STEPS` (visible in the training curve — flat top means timeouts).
  Consider lowering `MAX_STEPS`, making `REWARD_CONFIG["draw"]` more
  negative, or shrinking the per-action shaping rewards.
- **Win rate oscillates / declines:** Try reducing the learning rate or
  raising the batch size. Check `eval/win_rate` and `train/approx_kl`
  in TensorBoard — runaway KL means the policy is over-shooting updates.
- **Action-masking warning:** With `ACTION_SPACE = "flat_discrete"`
  the agent should rarely sample invalid actions; if it does, the masks
  are broken upstream.
- **Much better than expected:** You may have found better
  hyperparameters! Consider contributing them back.

### Next steps

1. **Graduate the opponent:** Set `OPPONENT = "simple"` once you've saturated `random`
2. **Try different maps:** Larger maps (10×10, 14×14) are harder
3. **Tune hyperparameters:** Adjust `ent_coef`, `learning_rate`, etc.
4. **Self-play training:** See `train/train_self_play.py`
5. **AlphaZero:** See `train/train_alphazero.py` for MCTS-based training


In [ ]:
# Clean up environments
vec_env.close()
eval_env.close()
print("Done.")
